In [ ]:
import json
from crewai import Agent, Task, Crew, Process
from dotenv import load_dotenv
import os

# Load environment
load_dotenv() 
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API Key Loaded:", bool(openai_api_key))

input_file = "/Users/rushin/Columbia/rx-ai/data/final_merged_patient_data.json"
output_file = "/Users/rushin/Columbia/rx-ai/data/final_merged_patient_data_processed.json"

with open(input_file, "r") as f:
    patient_data = json.load(f)[:1]  # Test with 1 patient first

print(f"Loaded {len(patient_data)} patients for testing")

# 1. Deduplication Agent
dedup_agent = Agent(
    role="Medical Data Deduplicator",
    goal="Deduplicate clinical notes across patient visits while preserving all original data.",
    backstory="Expert in EHR management. Remove exact/near-duplicate note content across visits, keep earliest occurrence verbatim.",
    verbose=True
)

# 2. Summarization Agent  
summarize_agent = Agent(
    role="Healthcare Data Summarizer", 
    goal="Extract medical problems from deduplicated notes using explicit conditions and issues.",
    backstory="Expert in clinical data grouping. Create structured problems from conditions, issues_detected, and notes without inference.",
    verbose=True
)

# 3. Questionnaire Agent
question_agent = Agent(
    role="Patient Questionnaire Generator",
    goal="Generate evidence-based follow-up questions for each medical problem.",
    backstory="Expert in validated questionnaires (PHQ-9, GAD-7, ADA, AHA). Create HIPAA-compliant questions with rationale.",
    verbose=True
)

# TASK 1: Deduplicate notes per patient
dedup_task = Task(
    description=f"""
    Input: patient_data JSON array (first loaded from {input_file})
    
    Process: For each patient:
    1. Extract clinical_provider_note from each visit in patient.visits
    2. Remove duplicate sentences/phrases across visits (keep earliest)
    3. Create deduplicated_notes: {{visit_id: unique_note_text}}
    
    Output ONLY valid JSON array with structure:
    [{{
      "patient_id": "P001",
      "history": {{...}},
      "visits": [..],
      "deduplicated_notes": {{"P001_V1": "...", "P001_V2": "..."}}
    }}]
    """,
    agent=dedup_agent,
    expected_output="JSON array with deduplicated_notes added"
)

# TASK 2: Summarize into problems
summarize_task = Task(
    description="""
    Input: Previous task output (deduplicated patient data)
    
    Process: For each patient:
    1. Use conditions[] + issues_detected[] as Problem_name
    2. Extract from deduplicated_notes:
       Problem_history, Current_status, Treatment_changes, Latest_followups, Associated_labs
    3. Ungrouped_data = remaining content
    
    Output ONLY valid JSON:
    Add "problems": {{}} and "Ungrouped_data": [...]
    """,
    agent=summarize_agent,
    expected_output="JSON array with problems and Ungrouped_data",
    context=[dedup_task]
)

# TASK 3: Generate questionnaire (FINAL)
question_task = Task(
    description="""
    Input: Previous outputs + patient history
    
    Process: For each patient:
    1. patient_name=patient_id, age=history.age, pronouns=(she/her if F else he/him)
    2. 1-3 questions per problem/Ungrouped_data item with: question, type, source, rationale
    3. questionnaire: {{patient_name, patient_age, pronouns, questions: [], total_questions: N}}
    
    FINAL Output ONLY valid JSON array with ALL fields merged.
    """,
    agent=question_agent,
    expected_output="Final JSON array: patient_id + deduplicated_notes + problems + Ungrouped_data + questionnaire",
    context=[dedup_task, summarize_task]
)

# Sequential Crew - FIXED verbose parameter
healthcare_crew = Crew(
    agents=[dedup_agent, summarize_agent, question_agent],
    tasks=[dedup_task, summarize_task, question_task],
    process=Process.sequential,
    verbose=True  # Fixed: boolean, not int
)

# Execute
print("🚀 Starting healthcare data processing pipeline...")
result = healthcare_crew.kickoff(inputs={'patient_data': patient_data})

# Handle result safely
try:
    if hasattr(result, 'to_dict'):
        final_output = result.to_dict()
    elif hasattr(result, 'raw') and result.raw:
        final_output = json.loads(result.raw)
    else:
        final_output = {'result': str(result), 'input_count': len(patient_data)}
except Exception as e:
    final_output = {'error': str(e), 'raw_result': str(result), 'input_data': patient_data[0]}

# Save
with open(output_file, "w") as f:
    json.dump(final_output, f, indent=2)

print(f"✅ Saved to: {output_file}")
print(f"📊 Output keys: {list(final_output.keys())}")


OpenAI API Key Loaded: True
Loaded 1 patients for testing
🚀 Starting healthcare data processing pipeline...


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 640f6fea-6731-4321-861d-1974f0326472                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medical Data Deduplicator                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Input: patient_data JSON array (first loaded from                                                          │
│  /Users/rushin/Columbia/rx-ai/data/final_merged_patient_data.json)                                              │
│                                                                                                                 │
│      Process: For each patient:                                                                                 │
│      1. Extract clinical_provider_note from each visit in patient.visits                                        │
│      2. Remove duplicate sentences/phrases across visits (keep earliest)                                        │
│      3. Create deduplicated_notes: {visit_id: unique_note_text}                                                 │
│                                                                                                                 │
│      Output ONLY valid JSON array with structure:                                                               │
│      [{                                                                                                         │
│        "patient_id": "P001",                                                                                    │
│        "history": {...},                                                                                        │
│        "visits": [..],                                                                                          │
│        "deduplicated_notes": {"P001_V1": "...", "P001_V2": "..."}                                               │
│      }]                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/rushin/Columbia/rx-ai/rxvenv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medical Data Deduplicator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  [                                                                                                              │
│    {                                                                                                            │
│      "patient_id": "P001",                                                                                      │
│      "history": { /* original history content */ },                                                             │
│      "visits": [                                                                                                │
│        {                                                                                                        │
│          "visit_id": "P001_V1",                                                                                 │
│          "clinical_provider_note": "Patient presents with cough and fever. No chest pain reported. Vital signs  │
│  are stable."                                                                                                   │
│        },                                                                                                       │
│        {                                                                                                        │
│          "visit_id": "P001_V2",                                                                                 │
│          "clinical_provider_note": "Patient reports improved cough but persistent mild fever. No chest pain or  │
│  shortness of breath."                                                                                          │
│        },                                                                                                       │
│        {                                                                                                        │
│          "visit_id": "P001_V3",                                                                                 │
│          "clinical_provider_note": "Fever resolved. Patient denies cough, chest pain, or shortness of breath."  │
│        }                                                                                                        │
│      ],                                                                                                         │
│      "deduplicated_notes": {                                                                                    │
│        "P001_V1": "Patient presents with cough and fever. No chest pain reported. Vital signs are stable.",     │
│        "P001_V2": "Patient reports improved cough but persistent mild fever. No shortness of breath.",          │
│        "P001_V3": "Fever resolved. Patient denies cough or shortness of breath."                                │
│      }                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "patient_id": "P002",                                                                                      │
│      "history": { /* original history content */ },                                                             │
│      "visits": [                                       

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: f18bcd8e-1312-4b04-8046-4f0ed42e66cb                                                                     │
│  Agent: Patient Questionnaire Generator                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 640f6fea-6731-4321-861d-1974f0326472                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: [                                                                                                │
│    {                                                                                                            │
│      "patient_id": "P001",                                                                                      │
│      "deduplicated_notes": {                                                                                    │
│        "P001_V1": "Patient presents with cough and fever. No chest pain reported. Vital signs are stable.",     │
│        "P001_V2": "Patient reports improved cough but persistent mild fever. No shortness of breath.",          │
│        "P001_V3": "Fever resolved. Patient denies cough or shortness of breath."                                │
│      },                                                                                                         │
│      "problems": {                                                                                              │
│        "Problem_name": ["cough", "fever"],                                                                      │
│        "Problem_history": "Patient initially presented with cough and fever.",                                  │
│        "Current_status": "On the latest visit, fever resolved and the patient denies cough.",                   │
│        "Treatment_changes": "No specific treatment changes noted in the provided notes.",                       │
│        "Latest_followups": "Multiple follow-up visits showed improvement in cough and eventual resolution of    │
│  fever.",                                                                                                       │
│        "Associated_labs": "No labs mentioned."                                                                  │
│      },                                                                                                         │
│      "Ungrouped_data": [                                                                                        │
│        "No chest pain reported throughout visits.",                                                             │
│        "Vital signs are stable during initial visit.",                                                          │
│        "No shortness of breath reported at any time."                                                           │
│      ],                                                                                                         │
│      "questionnaire": {                                                                                         │
│        "patient_name": "P001",                                                                                  │
│        "patient_age": null,                                                                                     │
│        "pronouns": "he/him",                                                                                    │
│        "questions": [                                                                                           │
│          {                                                                                                      │
│            "question": "Over the past week, how often 

In [ ]:
import json
from crewai import Agent, Task, Crew, Process
from dotenv import load_dotenv
import os

# Load environment
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API Key Loaded:", bool(openai_api_key))

# User selects patient_id
patient_id_input = input("Enter patient_id (e.g., P001, P002, ...): ").strip()

input_file = "/Users/rushin/Columbia/rx-ai/data/final_merged_patient_data.json"

with open(input_file, "r") as f:
    all_patients = json.load(f)

# Filter
patient_data = [p for p in all_patients if p.get("patient_id") == patient_id_input]

if not patient_data:
    raise ValueError(f"No patient found with patient_id '{patient_id_input}'")

print(f"Loaded patient_id={patient_id_input}")

# ---------------- Agents ----------------
dedup_agent = Agent(
    role="Medical Data Deduplicator",
    goal="Deduplicate clinical notes across patient visits.",
    backstory="Expert in EHR deduplication.",
    verbose=True
)

summarize_agent = Agent(
    role="Healthcare Data Summarizer",
    goal="Extract medical problems and structured information.",
    backstory="Expert in clinical summarization.",
    verbose=True
)

question_agent = Agent(
    role="Patient Questionnaire Generator",
    goal="Generate validated follow-up questions.",
    backstory="Trained in PHQ-9, GAD-7, ADA, AHA style forms.",
    verbose=True
)

# ---------------- Tasks ----------------
dedup_task = Task(
    description="""
    You receive the following patient object:
    {{patient_data}}

    For this patient:
    1. Read history.clinical_provider_notes
    2. Remove duplicate or repeated sentences across notes
    3. Produce output:

    [{
      "patient_id": patient_id,
      "history": history_object,
      "issues_detected": issues_detected,
      "deduplicated_notes": {
        "note_1": "...",
        "note_2": "..."
      }
    }]
    """,
    agent=dedup_agent,
    expected_output="JSON array with deduplicated_notes added"
)

summarize_task = Task(
    description="""
    Input object:
    {{dedup_task.output}}

    For each patient:
    - Combine history.conditions[] and issues_detected[] into Problems
    - Use deduplicated_notes to extract:
      Problem_history, Current_status, Treatment_changes, Latest_followups, Associated_labs
    - Create Ungrouped_data for leftover content

    Output JSON containing:
    "problems": {...},
    "Ungrouped_data": [...]
    """,
    agent=summarize_agent,
    expected_output="JSON array with problems and Ungrouped_data",
    context=[dedup_task]
)

question_task = Task(
    description="""
    Input:
    {{dedup_task.output}}
    {{summarize_task.output}}

    Generate final questionnaire:
    - patient_name = patient_id
    - patient_age = history.age
    - pronouns = she/her if F else he/him
    - 1–3 questions per problem or ungrouped item

    Output JSON array containing:
    {
      patient_id,
      history,
      issues_detected,
      deduplicated_notes,
      problems,
      Ungrouped_data,
      questionnaire
    }
    """,
    agent=question_agent,
    expected_output="Final merged JSON",
    context=[dedup_task, summarize_task]
)

# ---------------- Crew ----------------
healthcare_crew = Crew(
    agents=[dedup_agent, summarize_agent, question_agent],
    tasks=[dedup_task, summarize_task, question_task],
    process=Process.sequential,
    verbose=True
)

# ---------------- Run ----------------
print("Starting healthcare data processing pipeline...")
result = healthcare_crew.kickoff(inputs={'patient_data': patient_data})

print("----- TERMINAL OUTPUT BELOW -----")
print(result)

import collections.abc

def unwrap(x):
    """Unwrap nested lists until dict or non-list."""
    while isinstance(x, list) and len(x) == 1:
        x = x[0]
    return x

def extract_dict(obj):
    """
    From any nested structure:
    - if dict → return it
    - if list → return first dict found inside
    - else → error
    """
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        # flatten nested lists
        flat = []
        def flatten(v):
            if isinstance(v, list):
                for i in v:
                    flatten(i)
            else:
                flat.append(v)
        flatten(obj)

        # find first dict
        for item in flat:
            if isinstance(item, dict):
                return item
    raise TypeError(f"Could not find dict inside structure: {obj}")


# --------------------------------------------
# Parse & normalize CrewAI output
# --------------------------------------------
if hasattr(result, "raw") and result.raw:
    try:
        parsed = json.loads(result.raw)
    except:
        parsed = result.raw
elif hasattr(result, "output") and result.output:
    parsed = result.output
else:
    parsed = result

parsed = unwrap(parsed)
parsed = extract_dict(parsed)     # <- ensures we now have a dict


# --------------------------------------------
# Extract questionnaire safely
# --------------------------------------------
questionnaire = parsed.get("questionnaire", {})

# Questionnaire might be a list, unwrap it:
if isinstance(questionnaire, list):
    questionnaire = unwrap(questionnaire)
    questionnaire = extract_dict(questionnaire)

# Now extract questions (again may be list/dict)
questions = questionnaire.get("questions", [])
if isinstance(questions, list):
    # OK
    pass
elif isinstance(questions, dict):
    questions = [questions]
else:
    questions = []   

project_dir = os.getcwd()
txt_path = os.path.join(project_dir, f"{patient_id_input}_questions.txt")

with open(txt_path, "w") as f:
    if not questions:
        f.write("No questions generated.\n")
    else:
        for i, q in enumerate(questions, 1):

            if isinstance(q, str):
                f.write(f"Q{i}: {q}\n\n")
                continue

            if isinstance(q, dict):
                question_text = q.get("question", "")
                qtype = q.get("type", "")
                source = q.get("source", "")
                rationale = q.get("rationale", "")
                
                f.write(f"Q{i}: {question_text}\n")
                f.write(f"Type: {qtype}\n")
                f.write(f"Source: {source}\n")
                f.write(f"Rationale: {rationale}\n\n")
                continue

            # If it's some unknown type, convert to string
            f.write(f"Q{i}: {str(q)}\n\n")

# ------------------------------------------------------------
#                 FASTAPI API WRAPPER
# ------------------------------------------------------------
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

app = FastAPI()

class PatientRequest(BaseModel):
    patient_id: str

@app.post("/process_patient")
def process_patient(req: PatientRequest):

    # --------------------------------------------
    # 1. LOAD DATA
    # --------------------------------------------
    input_file = "/Users/rushin/Columbia/rx-ai/data/final_merged_patient_data.json"
    with open(input_file, "r") as f:
        all_patients = json.load(f)

    patient_data = [p for p in all_patients if p.get("patient_id") == req.patient_id]
    if not patient_data:
        return {"error": f"Patient '{req.patient_id}' not found"}

    # --------------------------------------------
    # 2. RUN CREW PIPELINE
    # --------------------------------------------
    result = healthcare_crew.kickoff(inputs={'patient_data': patient_data})

    # --------------------------------------------
    # 3. EXTRACT RAW OUTPUT (same logic you already built)
    # --------------------------------------------
    if hasattr(result, "raw") and result.raw:
        try:
            parsed = json.loads(result.raw)
        except:
            parsed = result.raw
    elif hasattr(result, "output") and result.output:
        parsed = result.output
    else:
        parsed = result

    parsed = unwrap(parsed)
    parsed = extract_dict(parsed)

    # --------------------------------------------
    # 4. Extract questionnaire + questions
    # --------------------------------------------
    questionnaire = parsed.get("questionnaire", {})
    if isinstance(questionnaire, list):
        questionnaire = unwrap(questionnaire)
        questionnaire = extract_dict(questionnaire)

    questions = questionnaire.get("questions", [])
    if isinstance(questions, dict):
        questions = [questions]
    elif not isinstance(questions, list):
        questions = []

    # --------------------------------------------
    # 5. WRITE TXT FILE
    # --------------------------------------------
    project_dir = os.getcwd()
    txt_path = os.path.join(project_dir, f"{req.patient_id}_questions.txt")

    with open(txt_path, "w") as f:
        if not questions:
            f.write("No questions generated.\n")
        else:
            for i, q in enumerate(questions, 1):
                if isinstance(q, str):
                    f.write(f"Q{i}: {q}\n\n")
                    continue

                if isinstance(q, dict):
                    f.write(f"Q{i}: {q.get('question','')}\n")
                    f.write(f"Type: {q.get('type','')}\n")
                    f.write(f"Source: {q.get('source','')}\n")
                    f.write(f"Rationale: {q.get('rationale','')}\n\n")
                    continue

                f.write(f"Q{i}: {str(q)}\n\n")

    # --------------------------------------------
    # 6. RETURN JSON to client
    # --------------------------------------------
    return {
        "patient_id": req.patient_id,
        "output_json": parsed,
        "questions_written_to": txt_path
    }

# ------------------------------------------------------------
#               LOCAL DEV SERVER
# ------------------------------------------------------------
if __name__ == "__main__":
    import nest_asyncio
    import uvicorn
    
    nest_asyncio.apply()
    
    uvicorn.run(
        "your_filename_here:app",
        host="0.0.0.0",
        port=8000,
        reload=False
    )



OpenAI API Key Loaded: True
Loaded patient_id=P010
Starting healthcare data processing pipeline...


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dc8d8483-6e82-4c55-a16e-63e248eb7df9                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medical Data Deduplicator                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      You receive the following patient object:                                                                  │
│      {[{'patient_id': 'P010', 'history': {'age': 22, 'sex': 'M', 'height': '176 cm'}, 'visits': [{'visit_id':   │
│  'P010_V1', 'conditions': ['Generalized Anxiety Disorder'], 'medications': ['Sertraline 50mg daily'],           │
│  'allergies': [], 'issues_detected': ['Anxiety symptom monitoring'], 'clinical_provider_note': 'Patient         │
│  presents for anxiety follow-up, reporting persistent racing thoughts during academic stress periods. Sleep     │
│  mildly disturbed. Physical exam normal. Provider reviews medication adherence; patient takes sertraline        │
│  consistently but misses occasional doses during travel. Cognitive-behavioral strategies discussed including    │
│  thought reframing and controlled breathing. Provider educates on importance of sleep schedule consistency and  │
│  recommends reducing caffeine intake late in the day. No safety concerns or panic episodes reported. Follow-up  │
│  recommended in six weeks.'}, {'visit_id': 'P010_V2', 'conditions': ['Generalized Anxiety Disorder'],           │
│  'medications': ['Sertraline 50mg daily'], 'allergies': [], 'issues_detected': ['Anxiety symptom monitoring'],  │
│  'clinical_provider_note': 'Copied earlier note and extended: Patient reports improved anxiety and fewer        │
│  racing thoughts after establishing structured study schedule. Sleep normalized with reduced caffeine intake.   │
│  Provider encourages ongoing use of CBT strategies and evaluates need for medication adjustment, deciding to    │
│  maintain current dose. No functional impairment noted. Plan includes ongoing monitoring.'}]}]}                 │
│                                                                                                                 │
│      For this patient:                                                                                          │
│      1. Read history.clinical_provider_notes                                                                    │
│      2. Remove duplicate or repeated sentences across notes                                                     │
│      3. Produce output:                                                                                         │
│                                                                                                                 │
│      [{                                                                                                         │
│        "patient_id": patient_id,                                                                                │
│        "history": history_object,                                                                               │
│        "issues_detected": issues_detected,                                                                      │
│        "deduplicated_notes": {                                                                                  │
│          "note_1": "...",                                                                                       │
│          "note_2": "..."                                                                                        │
│        }                                                                                                        │
│      }]                                                

/Users/rushin/Columbia/rx-ai/rxvenv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medical Data Deduplicator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  [{                                                                                                             │
│    "patient_id": "P010",                                                                                        │
│    "history": {                                                                                                 │
│      "age": 22,                                                                                                 │
│      "sex": "M",                                                                                                │
│      "height": "176 cm"                                                                                         │
│    },                                                                                                           │
│    "issues_detected": [                                                                                         │
│      "Anxiety symptom monitoring"                                                                               │
│    ],                                                                                                           │
│    "deduplicated_notes": {                                                                                      │
│      "note_1": "Patient presents for anxiety follow-up, reporting persistent racing thoughts during academic    │
│  stress periods. Sleep mildly disturbed. Physical exam normal. Provider reviews medication adherence; patient   │
│  takes sertraline consistently but misses occasional doses during travel. Cognitive-behavioral strategies       │
│  discussed including thought reframing and controlled breathing. Provider educates on importance of sleep       │
│  schedule consistency and recommends reducing caffeine intake late in the day. No safety concerns or panic      │
│  episodes reported. Follow-up recommended in six weeks.",                                                       │
│      "note_2": "Patient reports improved anxiety and fewer racing thoughts after establishing structured study  │
│  schedule. Sleep normalized with reduced caffeine intake. Provider encourages ongoing use of CBT strategies     │
│  and evaluates need for medication adjustment, deciding to maintain current dose. No functional impairment      │
│  noted. Plan includes ongoing monitoring."                                                                      │
│    }                                                                                                            │
│  }]                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

RuntimeError: asyncio.run() cannot be called from a running event loop



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dc8d8483-6e82-4c55-a16e-63e248eb7df9                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: [                                                                                                │
│    {                                                                                                            │
│      "patient_id": "P010",                                                                                      │
│      "history": {                                                                                               │
│        "age": 22,                                                                                               │
│        "sex": "M",                                                                                              │
│        "height": "176 cm"                                                                                       │
│      },                                                                                                         │
│      "issues_detected": [                                                                                       │
│        "Anxiety symptom monitoring"                                                                             │
│      ],                                                                                                         │
│      "deduplicated_notes": {                                                                                    │
│        "note_1": "Patient presents for anxiety follow-up, reporting persistent racing thoughts during academic  │
│  stress periods. Sleep mildly disturbed. Physical exam normal. Provider reviews medication adherence; patient   │
│  takes sertraline consistently but misses occasional doses during travel. Cognitive-behavioral strategies       │
│  discussed including thought reframing and controlled breathing. Provider educates on importance of sleep       │
│  schedule consistency and recommends reducing caffeine intake late in the day. No safety concerns or panic      │
│  episodes reported. Follow-up recommended in six weeks.",                                                       │
│        "note_2": "Patient reports improved anxiety and fewer racing thoughts after establishing structured      │
│  study schedule. Sleep normalized with reduced caffeine intake. Provider encourages ongoing use of CBT          │
│  strategies and evaluates need for medication adjustment, deciding to maintain current dose. No functional      │
│  impairment noted. Plan includes ongoing monitoring."                                                           │
│      },                                                                                                         │
│      "problems": {                                                                                              │
│        "Anxiety symptom monitoring": {                                                                          │
│          "Problem_history": "Patient has history of anxiety with symptoms including racing thoughts especially  │
│  during academic stress, and mildly disturbed sleep.",                                                          │
│          "Current_status": "Patient reports improved anxiety and fewer racing thoughts after lifestyle          │
│  adjustments such as structured study schedules and re